In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (83).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (61).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (55).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (127).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (167).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (67).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (53).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (34).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (173).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (4).jpg
/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val/Snow-Covered/Snow (45).jpg
/kaggle/input/datas

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision.models import mobilenet_v2
from torch.utils.data import DataLoader

In [3]:
from torchvision import transforms

IMG_SIZE = 224

def get_train_transform():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

def get_val_transform():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

def get_demo_transform():
    return get_val_transform()

In [4]:
TRAIN_DIR = "/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/train"
VAL_DIR   = "/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/val"

In [21]:
BATCH_SIZE = 64
EPOCHS = 25
LR = 0.001

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [23]:
train_dataset = datasets.ImageFolder(
    TRAIN_DIR, transform=get_train_transform()
)
val_dataset = datasets.ImageFolder(
    VAL_DIR, transform=get_val_transform()
)

In [24]:
class_names = train_dataset.classes
num_classes = len(class_names)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [26]:
model = mobilenet_v2(weights=None)
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)

In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [28]:
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss/len(train_loader):.4f}")

Epoch [1/25] Loss: 1.6533
Epoch [2/25] Loss: 1.3897
Epoch [3/25] Loss: 1.1898
Epoch [4/25] Loss: 1.0974
Epoch [5/25] Loss: 1.0353
Epoch [6/25] Loss: 0.9798
Epoch [7/25] Loss: 0.8928
Epoch [8/25] Loss: 0.8488
Epoch [9/25] Loss: 0.9059
Epoch [10/25] Loss: 0.8197
Epoch [11/25] Loss: 0.7196
Epoch [12/25] Loss: 0.7442
Epoch [13/25] Loss: 0.7429
Epoch [14/25] Loss: 0.6554
Epoch [15/25] Loss: 0.6363
Epoch [16/25] Loss: 0.6003
Epoch [17/25] Loss: 0.5724
Epoch [18/25] Loss: 0.5502
Epoch [19/25] Loss: 0.4669
Epoch [20/25] Loss: 0.4988
Epoch [21/25] Loss: 0.5894
Epoch [22/25] Loss: 0.4401
Epoch [23/25] Loss: 0.3914
Epoch [24/25] Loss: 0.3635
Epoch [25/25] Loss: 0.3666


In [29]:
torch.save(model.state_dict(), "solar_defect_model.pth")
print("Training completed & model saved.")

Training completed & model saved.


In [30]:
!dir

solar_defect_model.pth


In [35]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 87.0 MB/s eta 0:00:00:00:01:01


In [32]:
import torch
from torchvision.models import mobilenet_v2

IMG_SIZE = 224
NUM_CLASSES = 6  # Bird-drop, Clean, Dusty, Electrical-Damage, Physical-Damage, Snow-Covered

model = mobilenet_v2(weights=None)
model.classifier[1] = torch.nn.Linear(model.last_channel, NUM_CLASSES)

model.load_state_dict(torch.load("solar_defect_model.pth", map_location="cpu"))
model.eval()

dummy_input = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)

torch.onnx.export(
    model,
    dummy_input,
    "solar_defect_model.onnx",
    export_params=True,
    opset_version=18,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

print("ONNX model exported successfully.")

/tmp/ipykernel_55/1363127790.py:15: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0414 11:07:38.300000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0414 11:07:38.301000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0414 11:07:38.303000 55 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Tre

[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 105 of general pattern rewrite rules.
ONNX model exported successfully.


In [33]:
!dir

solar_defect_model.onnx  solar_defect_model.onnx.data  solar_defect_model.pth


In [40]:
import onnxruntime as ort
import numpy as np
from PIL import Image
import torchvision.transforms as transforms
import sys

MODEL_PATH = "solar_defect_model.onnx"
IMAGE_PATH = "/kaggle/input/datasets/alicjalena/pv-panel-defect-dataset/test/Snow-Covered/Snow (7).jpg"

CLASS_NAMES = [
    "Bird-drop",
    "Clean",
    "Dusty",
    "Electrical-Damage",
    "Physical-Damage",
    "Snow-Covered"
]

IMG_SIZE = 224

session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image = Image.open(IMAGE_PATH).convert("RGB")
image_tensor = transform(image).unsqueeze(0).numpy()

outputs = session.run([output_name], {input_name: image_tensor})

exp_scores = np.exp(outputs[0])
probs = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

pred_idx = np.argmax(probs)
confidence = probs[0][pred_idx]

print("Predicted Defect:", CLASS_NAMES[pred_idx])
print("Confidence:", round(float(confidence) * 100, 2), "%")

Predicted Defect: Snow-Covered
Confidence: 99.11 %


In [41]:
from IPython.display import FileLink

FileLink('solar_defect_model.onnx')

/kaggle/working/solar_defect_model.onnx